<a href="https://colab.research.google.com/github/INKARAN001/Designing-a-Fault-Tolerant-Distributed-Payment-processing-System/blob/main/SE3082_Lab10_IT24102182.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv


Fri Oct  9 09:58:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // each block runs only one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;              // host copies of a, b, c
    int *d_a, *d_b, *d_c;        // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate space for device copies of a, b, c
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate space for host copies of a, b, c and set up input values
    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // Copy inputs to device
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // initialize timer
    dStart = cpuSecond();

    // Launch add() kernel on GPU with N blocks
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock

    dElaps = cpuSecond() - dStart;

    // Copy result back to host
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Check the GPU's answers against the CPU
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex1_timed.cu


In [3]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed


83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0006161


In [4]:
cudaDeviceSynchronize();    // wait for the GPU to finish, THEN stop the clock


SyntaxError: invalid syntax (3462083566.py, line 1)

In [5]:
%%writefile ex1_nosync.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // cudaDeviceSynchronize();   <-- REMOVED for the experiment
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);


Writing ex1_nosync.cu


In [6]:
!nvcc -arch=sm_75 ex1_nosync.cu -o ex1_nosync && ./ex1_nosync


At end of source: error: expected a "}"
ex1_nosync.cu(22): note #3196-D: to match this "{"
  int main(void) {
                 ^

1 error detected in the compilation of "ex1_nosync.cu".


In [7]:
%%writefile ex1_nosync.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // cudaDeviceSynchronize();   <-- REMOVED for the experiment
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Overwriting ex1_nosync.cu


In [8]:
!nvcc -arch=sm_75 ex1_nosync.cu -o ex1_nosync && ./ex1_nosync


83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000901


Experiment – removing cudaDeviceSynchronize()

With cudaDeviceSynchronize(): Time taken = 0.0006161 s, 0 mismatches
Without cudaDeviceSynchronize(): Time taken = 0.0000901 s, 0 mismatches

The time dropped about 7 times because a kernel launch is asynchronous. The CPU starts the kernel and immediately continues, so the clock stops while the GPU is still working. Only the launch is measured, not the kernel work.

The answer still verifies correctly because cudaMemcpy (device to host) waits for the kernel to finish before copying, so the CPU always receives the finished results.


In [9]:
%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 3; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex3_blocks_timed.cu


In [10]:
!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed && ./ex3_blocks_timed && ./ex3_blocks_timed && ./ex3_blocks_timed


83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001400
83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001462
83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001230


In [11]:
%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 3; r++)
        printf("%d) %d + %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex4_threads_timed.cu


In [12]:
!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed && ./ex4_threads_timed && ./ex4_threads_timed && ./ex4_threads_timed


0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001609
0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001130
0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
Verification: 0 mismatches out of 512
Time taken - 0.0001290


In [13]:
%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // last 3 results (Lab 9 printed the last 1000)
    for (int r = N - 3; r < N; r++)
        printf("%d) %d x %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex5_vecmul_timed.cu


In [14]:
!nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed && ./ex5_vecmul_timed && ./ex5_vecmul_timed && ./ex5_vecmul_timed


Launching 19532 blocks x 512 threads
9999997) 24 x 63 = 1512
9999998) 7 x 69 = 483
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006421
Launching 19532 blocks x 512 threads
9999997) 24 x 63 = 1512
9999998) 7 x 69 = 483
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006189
Launching 19532 blocks x 512 threads
9999997) 24 x 63 = 1512
9999998) 7 x 69 = 483
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006170


In [19]:
%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks((COLS + 31) / 32, (ROWS + 15) / 16);
    printf("Grid: %d x %d blocks\n", numBlocks.x, numBlocks.y);

    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int col = COLS - 3; col < COLS; col++) {
        size_t idx = (size_t)(ROWS - 1) * COLS + col;
        printf("C[%d][%d] = %d x %d = %d\n", ROWS - 1, col, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Overwriting ex6_matrix_timed.cu


In [20]:
!nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed && ./ex6_matrix_timed && ./ex6_matrix_timed && ./ex6_matrix_timed


Grid: 313 x 625 blocks
C[9999][9997] = 69 x 82 = 5658
C[9999][9998] = 9 x 37 = 333
C[9999][9999] = 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0050781
Grid: 313 x 625 blocks
C[9999][9997] = 69 x 82 = 5658
C[9999][9998] = 9 x 37 = 333
C[9999][9999] = 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0050318
Grid: 313 x 625 blocks
C[9999][9997] = 69 x 82 = 5658
C[9999][9998] = 9 x 37 = 333
C[9999][9999] = 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0050550


Exercise 1(b) – Results

| Lab 9 program | Launch configuration | Run 1 (s) | Run 2 (s) | Run 3 (s) |
|---|---|---|---|---|
| Exercise 3 | 512 blocks × 1 thread | 0.0001400 | 0.0001462 | 0.0001230 |
| Exercise 4 | 1 block × 512 threads | 0.0001609 | 0.0001130 | 0.0001290 |
| Exercise 5 | 10,000,000 elements, 19,532 blocks × 512 threads | 0.0006421 | 0.0006189 | 0.0006170 |
| Exercise 6 | 100,000,000 elements, 313 × 625 blocks of 32 × 16 | 0.0050781 | 0.0050318 | 0.0050550 |

Prediction 1 – Ex 3 vs Ex 4:
I expected Exercise 4 (1 block × 512 threads) to be faster. The GPU runs threads in warps of 32, so a block with only 1 thread wastes 31 of the 32 lanes in its warp. Exercise 3 needs 512 warps for 512 additions, while Exercise 4 needs only 16 full warps.
In the measurements both average about 0.00013 s, so they are practically the same. With only 512 additions the real work is tiny, and the time is mostly the fixed cost of launching the kernel and synchronising. The warp inefficiency would only show with much larger data.

Prediction 2 – Ex 6 vs Ex 5:
Exercise 6 has 10× as many elements, so I expected about 10× the time. Measured: about 0.00505 s vs 0.00063 s, which is about 8×. This is close to 10×, because both kernels are limited by memory bandwidth (reading a and b, writing c), so the time grows with the amount of data. It is a bit less than 10× because the fixed launch overhead is a larger share of the smaller Exercise 5 time.


In [21]:
%%writefile ex5_big_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double totalStart, totalElaps, kStart, kElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);

    // whole sequence timer starts BEFORE the first copy
    totalStart = cpuSecond();

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // kernel-only timer
    kStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    kElaps = cpuSecond() - kStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // whole sequence timer stops AFTER the last copy
    totalElaps = cpuSecond() - totalStart;

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess)
        printf("Kernel launch failed: %s\n", cudaGetErrorString(err));

    for (int r = N - 3; r < N; r++)
        printf("%d) %d x %d = %d\n", r, a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Kernel only time    - %.7f\n", kElaps);
    printf("Whole sequence time - %.7f\n", totalElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}


Writing ex5_big_timed.cu


In [22]:
!nvcc -arch=sm_75 ex5_big_timed.cu -o ex5_big_timed && ./ex5_big_timed


Launching 195313 blocks x 512 threads
99999997) 69 x 82 = 5658
99999998) 9 x 37 = 333
99999999) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Kernel only time    - 0.0047920
Whole sequence time - 0.4518290


In [23]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Checks the value returned by a CUDA runtime call. On failure it prints the
// file, line number, error code and reason, then stops the program.
#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                               \
        if (error != cudaSuccess) {                                     \
            printf("Error: %s:%d, ", __FILE__, __LINE__);               \
            printf("code:%d, reason: %s\n", error,                      \
                   cudaGetErrorString(error));                          \
            exit(1);                                                    \
        }                                                               \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;              // host copies of a, b, c
    int *d_a, *d_b, *d_c;        // device copies of a, b, c
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, N);
    b = (int *)malloc(size); random_ints(b, N);
    c = (int *)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}


Writing ex2_check.cu


In [24]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check


83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0002549


In [25]:
!sed 's/add<<<N, 1>>>/add<<<N, 2048>>>/' ex2_check.cu > ex2_expA.cu && nvcc -arch=sm_75 ex2_expA.cu -o ex2_expA && ./ex2_expA


Error: ex2_expA.cu:57, code:1, reason: invalid argument


In [26]:
!sed 's/&d_a, size)/\&d_a, (size_t)1 << 50)/' ex2_check.cu > ex2_expB.cu && nvcc -arch=sm_75 ex2_expB.cu -o ex2_expB && ./ex2_expB


Error: ex2_expB.cu:42, code:2, reason: out of memory


In [27]:
!sed 's/add<<<N, 1>>>/add<<<N, 2048>>>/' ex1_timed.cu > ex1_expC.cu && nvcc -arch=sm_75 ex1_expC.cu -o ex1_expC && ./ex1_expC


83 + 77 = 0
86 + 34 = 0
77 + 90 = 0
15 + 26 = 0
93 + 24 = 0
Verification: 512 mismatches out of 512
Time taken - 0.0001268


Exercise 2 – Break it on purpose: results

Experiment A – too many threads (add<<<N, 2048>>>)
Output: Error: ex2_expA.cu:57, code:1, reason: invalid argument
The program stopped at line 57, which is CHECK(cudaGetLastError()) straight after the launch. The limit is 1024 threads per block, so the launch was rejected. The lab sheet expected code 9 ("invalid configuration argument"); the CUDA 13 runtime on this Colab GPU reports the same problem as code 1 ("invalid argument"). Either way the CHECK macro caught the failed launch.
Row of the table tested: "The kernel launch is wrong" – caught by CHECK(cudaGetLastError()).

Experiment B – impossible allocation (cudaMalloc of 1 << 50 bytes = 1 PB)
Output: Error: ex2_expB.cu:42, code:2, reason: out of memory
The program stopped at line 42, the first cudaMalloc, because the GPU has only about 15 GB.
Row of the table tested: "Any CUDA call that returns a value" – caught by wrapping cudaMalloc in CHECK(...).

Experiment C – too many threads without CHECK (Exercise 1 program)
Output: 83 + 77 = 0, 86 + 34 = 0, ... Verification: 512 mismatches out of 512
No error message was printed. The kernel never ran, so c stayed 0, but the program continued as if nothing was wrong and even printed a time. The only sign of the problem was the verification loop. Without that loop the wrong answers would go unnoticed. This is why every CUDA call should be checked with CHECK().


Exercise 3 – Scaling a picture with a 2D grid


In [28]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH  800              // columns
#define HEIGHT 600              // rows

#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                               \
        if (error != cudaSuccess) {                                     \
            printf("Error: %s:%d, ", __FILE__, __LINE__);               \
            printf("code:%d, reason: %s\n", error,                      \
                   cudaGetErrorString(error));                          \
            exit(1);                                                    \
        }                                                               \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Kernel from the lecture slide "Source Code of a PictureKernel", adapted
// for 8-bit pixels: scale every pixel value by 2.0, but never exceed 255.
__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
                              int height, int width) {
    // Calculate the row # of the d_Pin and d_Pout element
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    // Calculate the column # of the d_Pin and d_Pout element
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    // each thread computes one element of d_Pout if in range
    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;      // ONE flat array: 480,000 pixels
    size_t size  = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out;                // host: original and updated picture
    unsigned char *d_in, *d_out;                // device copies
    double dStart, dElaps;

    h_in  = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);
    CHECK(cudaMalloc((void **)&d_in,  size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: give every pixel of h_in a random value between 0 and 255
    for (size_t i = 0; i < count; i++)
        h_in[i] = rand() % 256;

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: 16 threads per block in BOTH x and y
    dim3 threadsPerBlock(16, 16);
    // TODO 3: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((WIDTH + 15) / 16, (HEIGHT + 15) / 16);
    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();
    // TODO 4: launch PictureKernel with <<<numBlocks, threadsPerBlock>>>
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: print the top-left 50 x 50 portion: first the original values,
    //         then the updated values
    printf("\nOriginal picture (top-left 50 x 50):\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++)
            printf("%3d ", h_in[r * WIDTH + c]);
        printf("\n");
    }


Writing ex3_picture.cu


In [29]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture


At end of source: error: expected a "}"
ex3_picture.cu(41): note #3196-D: to match this "{"
  int main(void) {
                 ^

1 error detected in the compilation of "ex3_picture.cu".


In [30]:
%%writefile -a ex3_picture.cu

    printf("\nUpdated picture (top-left 50 x 50):\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++)
            printf("%3d ", h_out[r * WIDTH + c]);
        printf("\n");
    }

    // TODO 6: check every pixel on the CPU (expected = 2 x original, capped at 255),
    //         count the mismatches, and count how many pixels were capped
    int errors = 0, capped = 0;
    for (size_t i = 0; i < count; i++) {
        int v = 2 * h_in[i];
        if (v > 255) { v = 255; capped++; }
        if (h_out[i] != v) errors++;
    }
    printf("\nVerification: %d mismatches out of %zu\n", errors, count);
    printf("Pixels capped at 255: %d\n", capped);

    printf("Time taken - %.7f\n", dElaps);

    free(h_in); free(h_out);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return 0;
}


Appending to ex3_picture.cu


In [31]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture


Grid: 50 x 38 blocks of 16 x 16 threads

Original picture (top-left 50 x 50):
103 198 105 115  81 255  74 236  41 205 186 171 242 251 227  70 124 194  84 248  27 232 231 141 118  90  46  99  51 159 201 154 102  50  13 183  49  88 163  90  37  93   5  23  88 233  94 212 171 178 
 96  34  10 167  77  49 181  91   3 160  13  34  13  71  93 205 155 135 120  86 213 112  76 156 134 234  15 152 242 235 156  83  13 167 250  90 216 176 181 219  80 194 253  93   9  90  42 165 226 163 
210 229 220  71  16 152 132 214 161  59  36  81  31  29 107 245  90 125  16 216  23 252 165  61 140  36 239 252 218 206  78 172 179  42 243 196 195 119 154 100 178 190 181 209 219  32 198  53 157 214 
158 212 187 189 162 172 238 192 141 116  54 141  24 225  34 225 154   4  34 178 109 178 216 130 145 231 176 222 132 115 155  34  71  86 223 233   2 205 169 143  65 224  28  90 193  63  59  91  67  93 
219 240 232   2 175 227 102  92  27 167 209 145 153  42 245 250 103 153 124 186 196 109  26  59 117 143  79  87 135 18

Exercise 3 – Results

Grid: 50 x 38 blocks of 16 x 16 threads (1,900 blocks, 486,400 threads for 480,000 pixels)
Verification: 0 mismatches out of 480000
Pixels capped at 255: 240,121

Prediction – how many pixels will be capped?
A pixel is capped when 2 × value > 255, i.e. for original values 128 to 255. That is 128 of the 256 possible values, so about half of the 480,000 pixels (about 240,000) should be capped. The program reported 240,121, which matches.

Grid size: 800 / 16 = 50 blocks in x (exact). 600 / 16 = 37.5, rounded up to 38 blocks in y. The last row of blocks has only 8 of its 16 rows inside the picture; the other 8 rows of threads are stopped by the guard (Row < height) && (Col < width).


Exercise 4 – Multiplying two 3D matrices element by element


In [32]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500                  // x: columns
#define NY 500                  // y: rows
#define NZ 500                  // z: planes (depth)

#define CHECK(call)                                                     \
    do {                                                                \
        const cudaError_t error = (call);                               \
        if (error != cudaSuccess) {                                     \
            printf("Error: %s:%d, ", __FILE__, __LINE__);               \
            printf("code:%d, reason: %s\n", error,                      \
                   cudaGetErrorString(error));                          \
            exit(1);                                                    \
        }                                                               \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Element-wise product: C[z][y][x] = A[z][y][x] * B[z][y][x]
__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // TODO 1: the x, y and z coordinates of this thread
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: only compute if x, y AND z are all inside the volume
    if (x < nx && y < ny && z < nz) {
        // TODO 3: flat index of (x, y, z): (z * ny + y) * nx + x
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}


Writing ex4_volume.cu


In [33]:
%%writefile -a ex4_volume.cu

int main(void) {
    size_t count = (size_t)NX * NY * NZ;        // 125,000,000 elements
    size_t size  = count * sizeof(int);         // 500 MB per array
    int *a, *b, *c;                             // host copies (flat arrays)
    int *d_a, *d_b, *d_c;                       // device copies
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size); random_ints(a, count);
    b = (int *)malloc(size); random_ints(b, count);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);
    // TODO 4: blocks needed in x, y and z, rounding UP
    dim3 numBlocks((NX + 7) / 8, (NY + 7) / 8, (NZ + 7) / 8);
    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();
    // TODO 5: launch mulElem3D with <<<numBlocks, threadsPerBlock>>>
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: print the last 1000 elements of C, in memory order
    for (size_t i = count - 1000; i < count; i++) {
        int z = i / ((size_t)NX * NY);
        int y = (i / NX) % NY;
        int x = i % NX;
        printf("C[%d][%d][%d] = %d x %d = %d\n", z, y, x, a[i], b[i], c[i]);
    }

    // TODO 7: check all elements on the CPU and print the number of mismatches
    size_t errors = 0;
    for (size_t i = 0; i < count; i++)
        if (c[i] != a[i] * b[i]) errors++;
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    printf("Time taken - %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}


Appending to ex4_volume.cu


In [34]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume


Grid: 63 x 63 x 63 blocks
C[499][498][0] = 76 x 15 = 1140
C[499][498][1] = 38 x 93 = 3534
C[499][498][2] = 34 x 11 = 374
C[499][498][3] = 85 x 47 = 3995
C[499][498][4] = 3 x 38 = 114
C[499][498][5] = 43 x 82 = 3526
C[499][498][6] = 16 x 52 = 832
C[499][498][7] = 28 x 23 = 644
C[499][498][8] = 32 x 28 = 896
C[499][498][9] = 80 x 98 = 7840
C[499][498][10] = 69 x 81 = 5589
C[499][498][11] = 45 x 79 = 3555
C[499][498][12] = 30 x 2 = 60
C[499][498][13] = 41 x 13 = 533
C[499][498][14] = 30 x 81 = 2430
C[499][498][15] = 30 x 74 = 2220
C[499][498][16] = 41 x 13 = 533
C[499][498][17] = 14 x 18 = 252
C[499][498][18] = 83 x 20 = 1660
C[499][498][19] = 13 x 28 = 364
C[499][498][20] = 86 x 17 = 1462
C[499][498][21] = 13 x 6 = 78
C[499][498][22] = 88 x 57 = 5016
C[499][498][23] = 71 x 29 = 2059
C[499][498][24] = 28 x 17 = 476
C[499][498][25] = 78 x 2 = 156
C[499][498][26] = 50 x 37 = 1850
C[499][498][27] = 86 x 40 = 3440
C[499][498][28] = 28 x 35 = 980
C[499][498][29] = 15 x 1 = 15
C[499][498][30] =

Exercise 4 – Results

Grid: 63 x 63 x 63 blocks of 8 x 8 x 8 threads (250,047 blocks, 128,024,064 threads for 125,000,000 elements)
First of the last 1000 results: C[499][498][0] = 76 x 15 = 1140
Last result: C[499][499][499] = 47 x 68 = 3196
Verification: 0 mismatches out of 125000000
Time taken: 0.0078869 s

THINK – 2D (Lab 9 Ex 6) vs 3D (this exercise)

| Kernel | Elements | Time (s) | Time per element |
|---|---|---|---|
| 2D, 32 x 16 blocks | 100,000,000 | ~0.00505 | ~0.050 ns |
| 3D, 8 x 8 x 8 blocks | 125,000,000 | 0.00789 | ~0.063 ns |

The 3D kernel has 1.25× the elements but took about 1.56× as long, so it is about 25% slower per element. Both kernels do the same simple work, so they are limited by memory bandwidth, not by computation. The arrangement of threads matters through how memory is accessed:
- In the 2D kernel, a warp of 32 threads covers 32 neighbouring columns of one row, so it reads one contiguous block of memory (fully coalesced).
- In the 3D kernel, a block is only 8 threads wide in x, so a warp of 32 threads covers 8 columns on 4 different rows. It reads 4 separate small pieces of memory, which uses memory bandwidth less efficiently.
- Also, 500 is not a multiple of 8, so 3D launches 2.4% extra idle threads, compared with only 0.16% for 2D.

So the arrangement matters little for correctness and for the amount of work, but it does affect speed because of memory coalescing. A block shape like (32, 4, 4) would keep the warps contiguous in x and should be closer to the 2D speed.
